In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

from pachner_traversal.utils import get_data_root

plt.rcParams.update(
    {
        "font.family": "serif",  # Matches typical LaTeX document fonts
        "font.size": 12,  # Base font size
        "axes.labelsize": 14,  # Larger axis labels
        "xtick.labelsize": 12,  # Tick sizes
        "ytick.labelsize": 12,
        "legend.fontsize": 12,
        "axes.grid": True,  # Turn on grid
        "grid.alpha": 0.4,  # Make grid faint so it doesn't distract
        "grid.linestyle": "--",  # Dashed grid lines
        "figure.dpi": 300,  # 300 DPI is standard for publication
        # 'text.usetex': True          # Uncomment if you have a full LaTeX engine installed
    }
)

In [4]:
data_root = get_data_root()
base_file_path = data_root / "results" / "sgd_models_dehydration" / "isoflop_scaling"

df = pd.read_csv(base_file_path / "all.csv", index_col=0)

df["comp"] = df.index.str.split("_").str[1].str.replace("pflop", "").astype(float)

df.columns = df.columns.str.strip()

In [5]:
def latex_sci(val: float, precision: int = 0) -> str:
    """Formats a float into standard LaTeX scientific notation: $a \\times 10^{b}$"""
    if val == 0:
        return "$0$"

    # Format to standard scientific representation first
    base, exp = f"{val:.{precision}e}".split("e")
    exp = int(exp)  # Strip leading zeros and plus signs

    return rf"${base} \times 10^{{{exp}}}$"

In [6]:
cmap = plt.colormaps["magma"]
colors = cmap(np.linspace(0, 1, 10))

fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
for i, comp in enumerate(np.sort(df["comp"].unique())):
    sdf = df[df["comp"] == comp]
    ax.scatter(
        sdf["n_params"],
        sdf["test_loss"],
        color=colors[i],
        linewidth=0.4,
        alpha=0.9,
        label=f"{latex_sci(comp * 1e15)}",
    )
    coefs = np.polyfit(np.log(sdf["n_params"]), sdf["test_loss"], 2)
    x_data = np.geomspace(sdf["n_params"].min(), sdf["n_params"].max(), 100)
    y_data = np.polyval(coefs, np.log(x_data))
    ax.plot(x_data, y_data, color=colors[i], linewidth=0.4, alpha=0.9, linestyle="--")


ax.set_xlabel(r"Parameters ($N$)")
ax.set_ylabel(r"Test Loss")


# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)
ax.legend(
    # loc="upper right",
    frameon=True,
    fontsize=10,
    title_fontsize=12,
)
ax.set_xscale("log")


plt.tight_layout()
plt.savefig("figures/isoflop_curves.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [7]:
fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
comps = np.sort(df["comp"].unique())
opt_params = []
for comp in comps:
    sdf = df[df["comp"] == comp]
    coefs = np.polyfit(np.log(sdf["n_params"]), sdf["test_loss"], 2)
    opt_params.append(np.exp(-coefs[1] / (2 * coefs[0])))


ax.scatter(
    comps * 1e15,
    opt_params,
    color="black",
    linewidth=0.4,
    alpha=0.9,
    # label=f"{latex_sci(comp * 1e15)}",
)

A = np.mean(np.log(opt_params) - np.log(comps) * 0.49)

x_data = np.geomspace(comps.min(), comps.max(), 100)
y_data = np.exp(np.polyval([0.49, A], np.log(x_data)))
ax.plot(x_data * 1e15, y_data, color="black", linewidth=0.4, alpha=0.9, linestyle="--")


ax.set_xlabel(r"Compute ($C$, FLOPs)")
ax.set_ylabel(r"Optimal Parameters ($N$)")

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_xscale("log")
ax.set_yscale("log")


plt.tight_layout()
plt.savefig("figures/fitted_optimal_parameters.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [8]:
fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
comps = np.sort(df["comp"].unique())
opt_loss = []
for comp in comps:
    sdf = df[df["comp"] == comp]
    coefs = np.polyfit(np.log(sdf["n_params"]), sdf["test_loss"], 2)
    opt_comp = np.exp(-coefs[1] / (2 * coefs[0]))
    opt_loss.append(np.polyval(coefs, np.log(opt_comp)))


ax.scatter(
    comps*1e15,
    opt_loss,
    color="black",
    linewidth=0.4,
    alpha=0.9,
    # label=f"{latex_sci(comp * 1e15)}",
)
opt_loss = np.array(opt_loss)
A = np.mean(np.log(opt_loss - 0.81) + np.log(comps) * 0.15)

x_data = np.geomspace(comps.min(), comps.max(), 100)
y_data = np.exp(np.polyval([-0.15, A], np.log(x_data))) + 0.81
ax.plot(x_data*1e15, y_data, color="black", linewidth=0.4, alpha=0.9, linestyle="--")


ax.set_xlabel(r"Compute ($C$, FLOPs)")
ax.set_ylabel(r"Optimal Loss $L(C)$")

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)
ax.set_xscale("log")
# ax.set_yscale("log")


plt.tight_layout()
plt.savefig("figures/fitted_optimal_loss.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [9]:
df = pd.read_csv('gen_efficiency.csv')

In [52]:
fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
ax.scatter(
    df["test_loss"],
    -np.log(df["spheres"] / 10_000),
    color="black",
    linewidth=0.4,
    alpha=0.9,
    # label=f"{latex_sci(comp * 1e15)}",
)

ax.invert_xaxis()
ax.set_xlabel(r"Test Loss")
ax.set_ylabel(r"Sphere Generation Shortfall (nats)")
# ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)


plt.tight_layout()
plt.savefig("figures/sphere_gen_frac.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [49]:
# import numpy as np
# from scipy.stats import beta


# def clopper_pearson_ci(k, n, confidence=0.95):
#     if n == 0:
#         return 0, 0
#     alpha = 1.0 - confidence
#     lower = 0.0 if k == 0 else beta.ppf(alpha / 2, k, n - k + 1)
#     upper = 1.0 if k == n else beta.ppf(1 - alpha / 2, k + 1, n - k)

#     return k/n - lower, upper - k/n

# clopper_pearson_ci = np.vectorize(clopper_pearson_ci)

# errors = clopper_pearson_ci(df["closed"] - df["spheres"], df["closed"])

In [48]:
# fig, ax = plt.subplots(figsize=(8, 5))

# # --- Main Plot ---
# ax.scatter(
#     df["test_loss"],
#     1 - df["spheres"] / df["closed"],
#     color="black",
#     linewidth=0.4,
#     alpha=0.9,
#     # label=f"{latex_sci(comp * 1e15)}",
# )
# ax.errorbar(df["test_loss"], 1 - df["spheres"] / df["closed"], yerr=errors, fmt='none', ecolor='gray', capsize=3, zorder=2, linewidth=0.4)
# ax.axhline(0.16, color='black', linewidth=0.4, alpha=0.9, linestyle="--")

# ax.invert_xaxis()
# ax.set_xlabel(r"Test Loss")
# ax.set_ylabel(r"Non-Sphere Closed Rate")
# ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# # Main plot grid (subtle)
# ax.grid(True, linestyle="--", alpha=0.3)


# plt.tight_layout()
# plt.savefig("figures/new_closed_frac.pdf", format="pdf", bbox_inches="tight")
# plt.clf()
# # plt.show()

In [53]:
import matplotlib.pyplot as plt
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.proportion import proportion_confint

k = df["spheres"].to_numpy(float)
n = df["closed"].to_numpy(float)
loss = df["test_loss"].to_numpy()
x = -loss                                   # fit variable (higher = better model)

# ---------- fit: logit P(sphere | closed) = b0 + b1 * x ----------
m = n > 0
fit = sm.GLM(np.c_[k[m], n[m] - k[m]], sm.add_constant(x[m]),
             family=sm.families.Binomial()).fit(scale="X2")

def shortfall(params, xv):                  # s = -ln sigmoid(b0 + b1 x)
    p = np.atleast_2d(params)
    return np.logaddexp(0, -(p[:, :1] + p[:, 1:2] * xv)).squeeze()

xs = np.linspace(x[m].min(), x[m].max(), 300)
s_fit = shortfall(fit.params, xs)
draws = np.random.default_rng(0).multivariate_normal(fit.params, fit.cov_params(), 5000)
band_lo, band_hi = np.percentile(shortfall(draws, xs), [2.5, 97.5], axis=0)

# ---------- points and Clopper-Pearson intervals, in nats ----------
ok = m & (k > 0)                            # zero spheres -> infinite shortfall, not plotted
p_hat = k[ok] / n[ok]
p_lo, p_hi = proportion_confint(k[ok], n[ok], alpha=0.05, method="beta")
s_hat = -np.log(p_hat)
s_lo = -np.log(p_hi)                        # upper p bound -> lower shortfall bound
s_hi = -np.log(np.clip(p_lo, 1e-12, None))  # lower p bound can be 0 -> clip
yerr = np.vstack([s_hat - s_lo, s_hi - s_hat])

# ---------- plot ----------
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(loss[ok], s_hat, yerr=yerr, fmt="none", ecolor="gray",
            capsize=3, linewidth=0.4, zorder=2)
ax.scatter(loss[ok], s_hat, color="black", s=14, zorder=3)
ax.fill_between(-xs, band_lo, band_hi, color="C3", alpha=0.2, lw=0, zorder=1)
ax.plot(-xs, s_fit, color="C3", linewidth=1.2, zorder=4)

ax.invert_xaxis()
ax.set_xlabel(r"Test Loss")
ax.set_ylabel(r"Topology Shortfall (nats)")
ax.set_ylim(0, 1.5)
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.savefig("figures/new_closed_frac.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [86]:
s_fit.max() - s_fit.min()

np.float64(0.08529971403355618)

In [27]:
class_path = (
    data_root
    / "results"
    / "sgd_models_dehydration"
    / "classification"
    / "classification_lr_0.0001"
)

disp_name = {
    "test_losses_s3": r"$S^3$",
    "test_losses_l72": r"$L(7, 2)$",
    "test_losses_rp3": r"$\mathbb{R}P^3$",
    "test_losses_random": r"Random Gluing",
    "train_losses": r"Train",
    "test_losses_l71": r"$L(7, 1)$",
    "test_losses_near_miss": r"Near Miss",
    "test_losses_s2s1": r"$S^2 \times S^1$",
    "test_losses_random_manifold": r"Random Manifold",
}

plot1 = [
    "train_losses",
    "test_losses_random_manifold",
    "test_losses_random",
    "test_losses_near_miss",
    "test_losses_s3",
]
plot2 = [
    "train_losses",
    "test_losses_l71",
    "test_losses_l72",
    "test_losses_s2s1",
    "test_losses_rp3",
]

all_data = {}
colors = plt.colormaps["okabe_ito"]

for f in class_path.iterdir():
    if f.is_file() and f.suffix == ".csv":
        data = pd.read_csv(f, index_col=0, header=None).squeeze()
        assert isinstance(data, pd.Series)
        all_data[disp_name[f.stem]] = data


for j, plot in enumerate([plot1, plot2]):
    fig, ax = plt.subplots(figsize=(8, 5))

    # --- Main Plot ---
    for i, raw_name in enumerate(plot):
        name = disp_name[raw_name]
        data = all_data[name]
        color = colors(i)

        ax.plot(
            data.index,
            data.values,
            color=color,
            # linewidth=0.4,
            alpha=0.9,
            label=name,
        )

    ax.set_xlabel(r"Iteration")
    ax.set_ylabel(r"Loss")
    ax.set_ylim((-0.05, 0.8))

    # Main plot grid (subtle)
    ax.grid(True, linestyle="--", alpha=0.3)
    ax.legend(
        # loc="upper right",
        frameon=True,
        fontsize=10,
        title_fontsize=12,
    )

    plt.tight_layout()
    plt.savefig(f"figures/classification_{j}.pdf", format="pdf", bbox_inches="tight")
    plt.clf()
    # plt.show()

<Figure size 2400x1500 with 0 Axes>

<Figure size 2400x1500 with 0 Axes>

In [28]:
ls_count = 2268
z_count = 1268
free2_count = 12
free_z_ls_count = 36
free_ls_ls_count = 51

labels = [
    r"$L(p, q)$",
    r"$S^2\times S^1$",
    r"$L(p, q)\# L(p', q')$",
    r"$L(p, q)\# S^2\times S^1$",
    r"$S^2\times S^1\# S^2\times S^1$",
]
values = [ls_count, z_count, free_ls_ls_count, free_z_ls_count, free2_count]

fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
ax.bar(labels, values)

ax.set_xlabel(r"Manifold Type")
ax.set_ylabel(r"Count of Occurrence")

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)


plt.tight_layout()
plt.savefig("figures/generated_type.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [29]:
df = pd.read_csv("ec.csv", index_col=0)

cmap = plt.colormaps["magma"]
colors = cmap(np.linspace(0, 1, 6))

fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
ax.scatter(df.index, df["-6"] / 10_000, label=r"$\chi\geq -6$", color=colors[0])
ax.scatter(df.index, df["-4"] / 10_000, label=r"$\chi\geq -4$", color=colors[1])
ax.scatter(df.index, df["-2"] / 10_000, label=r"$\chi\geq -2$", color=colors[2])
ax.scatter(df.index, df["0"] / 10_000, label=r"$\chi\geq 0$", color=colors[3])
ax.scatter(df.index, df["2"] / 10_000, label=r"$\chi\geq 2$", color=colors[4])

ax.invert_xaxis()
ax.set_xlabel(r"Test Loss")
ax.set_ylabel(r"Occurrence Rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)

ax.legend(
    frameon=True,
    fontsize=10,
    title_fontsize=12,
)


plt.tight_layout()
plt.savefig("figures/ec_threshold.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>

In [30]:
data = pd.read_csv("orientable.csv", index_col=0).squeeze()
assert isinstance(data, pd.Series)

fig, ax = plt.subplots(figsize=(8, 5))

# --- Main Plot ---
ax.scatter(data.index, data, color='black')

ax.invert_xaxis()
ax.set_xlabel(r"Test Loss")
ax.set_ylabel(r"Occurrence Rate (%)")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1.0))

# Main plot grid (subtle)
ax.grid(True, linestyle="--", alpha=0.3)


plt.tight_layout()
plt.savefig("figures/orientable.pdf", format="pdf", bbox_inches="tight")
plt.clf()
# plt.show()

<Figure size 2400x1500 with 0 Axes>